# 单通道：池化 trace 的闪烁特性

输入只有 `smTest_manual` 池化出来的 `TRACE_pooled/<组名>/traces_pooled.npz`，
**不碰影像**：读盘 → 看数据 → 定状态 → 统计 ON/OFF。

| 阶段 | 放在哪里 |
|---|---|
| 取 trace、人工挑、池化 | `smTest_batch` → `smTest_manual` |
| **闪烁统计（这一本）** | 只吃 `traces_pooled.npz` |
| 将来要加的分析工具（dwell 拟合等） | `smlm/trace/`，不放 notebook |

两条容易踩的坑（都已在库里堵住）：

1. **尾部 NaN**：池化为了对齐帧数，短的 trace 在后面补了 NaN。带着 NaN 跑
   `changepoint.segment` **不会报错**，但整条会被判成"一个暗段"。
   `tr.load_pooled` 已经按 `n_frames` 截断，读回来的长度就是真实帧数。
2. **每条的帧数不同**（1000 / 2000），所以任何统计都要按条处理，不能当成
   `(N, T)` 整齐矩阵用。


In [ ]:
# `import smlm` 需要 Py_Smile 在 sys.path 上; notebook 放在子目录里时自己找一下
import pathlib
import sys

for _q in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_q / 'smlm' / '__init__.py').is_file():
        sys.path.insert(0, str(_q))
        break

import matplotlib.pyplot as plt
import numpy as np

import smlm.trace as tr

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ======================= 输入 =======================
BASE_DIR = pathlib.Path(r"G:\SPARK\20261124")
POOL_DIR = BASE_DIR / 'TRACE_pooled'
GROUP = 'SPARK_6mW'                    # 先做这一组; 换组只改这里
POOLED = POOL_DIR / GROUP / 'traces_pooled.npz'

print('输入 :', POOLED)
print('存在 :', POOLED.is_file())


In [ ]:
traces, meta = tr.load_pooled(POOLED, verbose=True)

dt = meta['exposure_s']
n_frame = np.array([len(t['frames']) for t in traces])
peak = np.array([np.percentile(t['intensity'], 95) for t in traces])
em = np.array([t['emitter_id'] for t in traces])
fid = np.array([t['file_id'] for t in traces])

print(f'\n曝光 {dt:g} s, 池化数组宽 {meta["n_columns"]} 帧 '
      f'(= 最长的一条, 短的补 NaN)')
print(f'共 {len(traces)} 条 trace, 帧数 {n_frame.min()}~{n_frame.max()}')
for k, name in enumerate(meta['files']):
    m = fid == k
    print(f'  [{k}] {name:<44} {m.sum():>4} 条, '
          f'{n_frame[m].max() if m.any() else 0} 帧')
print(f'\n95% 分位亮度: 中位 {np.median(peak):.0f}, '
      f'范围 {peak.min():.0f}~{peak.max():.0f}')
print('前 3 条:', [int(v) for v in em[:3]])


## 先看一眼数据

按 95% 分位亮度降序（先看亮的）。左列是扣背景后的信号 `intensity`，右列是
`bg_flux`（孔径里的背景光子数，用来判断视野有没有被背景淹没）。


In [ ]:
order = np.argsort(-peak)[:12]

fig = tr.plot_trace_fields(
    traces, indices=order,
    groups={'intensity': ('intensity',),
            'bg_flux': dict(fields=('bg_flux',), color='tab:red')},
    xlabel=f'Frame  ({dt:g} s/frame)',
    title=f'{GROUP}   最亮的 {len(order)} 条')
plt.show()


一屏看完所有 trace：按亮度排序的 `intensity` 热图（横轴帧、纵轴 trace）。
帧数不同的那些在右边是**空白**，不是 0。图存到
`TRACE_pooled/<组名>/blink_heatmap.png`。


In [ ]:
def heatmap(traces, title=None, figsize=(10, 4)):
    """按亮度排序的一屏热图; 帧数不同 -> 短的在右边留白。"""
    order = np.argsort([-np.percentile(t['intensity'], 95) for t in traces])
    T = max(len(t['intensity']) for t in traces)
    M = np.full((len(traces), T), np.nan)
    for r, i in enumerate(order):
        M[r, :len(traces[i]['intensity'])] = traces[i]['intensity']
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(M, aspect='auto', cmap='magma', origin='lower',
                   interpolation='nearest')
    ax.set_xlabel('Frame')
    ax.set_ylabel('trace (sorted by brightness)')
    ax.set_title(title or '')
    fig.colorbar(im, ax=ax, label='intensity')
    fig.tight_layout()
    return fig


heatmap(traces, title=f'{GROUP}   n={len(traces)}').savefig(
    POOLED.parent / 'blink_heatmap.png', dpi=150,
    bbox_inches='tight', facecolor='white')
plt.show()


## 下一步（工具还没写进库里）

现在只有"干净的数据 + 看一眼"。要得到闪烁特性，接下来的工具应该放进
`smlm/trace/`，而不是留在本 notebook：

| 要做的事 | 落点 |
|---|---|
| 状态判定（给每条 trace 定 ON/OFF） | `ChangepointStates` 已就绪，`tr.apply_fillers` 可在**无影像**下直接跑 |
| 从 `state` 取连续 ON/OFF 段长 | 目前只有 `runs_of`；需要一个 `dwell_times(traces)` 把它们成对取出来（ON 段 = 亮态连续帧，OFF 段 = 相邻两个 ON 之间 + 开头） |
| 截断偏差校正 / 指数拟合 | 新增，`off` 段要从**存活区间**（未漂白的那段）里取，否则漂白后的长暗段会污染分布 |
| 汇总与作图 | 接 `plot_state_summary`，再补一个 dwell 分布图 |

**待定**：`bleach_frames` 取多少（现在 smTest 用 500 帧）、`vst` 开不开
（暗分子建议 `'anscombe'`，亮分子反而 raw 更好）、单帧 glitch 要不要
另外清理（`min_seg` 已经压掉单帧）。
